# 🛡️ SENTINEL-BELT: AI-Powered Multi-Gas Toxicity & Fall Detection System
### Confined-Space Life-Safety Telemetry & Machine Learning Validation Suite
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### 📌 Project Executive Summary
Every year, thousands of municipal sewer and manhole workers worldwide face life-threatening hazards from **invisible toxic gases** ($H_2S$, $CO$, $CH_4$), **oxygen-deficient atmospheres** ($<19.5\%$ $O_2$), **rapid water inundation**, and **traumatic slips/falls**. 

The **SENTINEL-BELT** is an edge-AI wearable safety system integrating:
1. **Multi-Gas Electrochemical & Catalytic Sensing**: $O_2$ (Galvanic), $H_2S$ (MQ-136), $CO$ (MQ-7), $CH_4$ (MQ-4).
2. **Sub-Surface Inundation Gauge**: Ultrasonic echo depth profiling for water rise.
3. **Motion & Vital Biometrics**: 6-axis IMU for fall detection and immobility monitoring + $SpO_2$/Pulse.
4. **Composite Risk Engine & Edge Machine Learning**: Random Forest multi-sensor fusion + Isolation Forest anomaly detection transpiled directly to ESP32-S3 C-firmware.
5. **Immediate On-Body Countermeasures**: Acoustic piezo siren, haptic vibration, high-contrast OLED, and solenoid-actuated emergency oxygen shower.

---

### 📐 Mathematical Safety Formulation
The real-time composite hazard score $R(t) \in [0, 100]$ is computed as:
$$R(t) = 100 \times \left( W_{O_2} S_{O_2} + W_{H_2S} S_{H_2S} + W_{CO} S_{CO} + W_{CH_4} S_{CH_4} + W_W S_W + W_M S_M \right)$$

Where:
- $W_{O_2} = 0.30$, $W_{H_2S} = 0.25$, $W_{CO} = 0.15$, $W_{CH_4} = 0.15$, $W_W = 0.10$, $W_M = 0.05$
- **Hard Safety Overrides**: If $O_2 < 19.5\%$ or Fall Detected $\implies$ Immediate **CRITICAL / EVACUATION** status regardless of other metrics.


## 1. Environment Setup & Dependency Installation
Let's install and verify the required numerical, visualization, and ML libraries.


In [ ]:
# Install dependencies if running on Google Colab or fresh environment
!pip install -q numpy pandas scikit-learn matplotlib seaborn joblib ipywidgets

import time
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve
)
from sklearn.preprocessing import label_binarize

# Configure publication aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cbd5e1'
plt.rcParams['axes.linewidth'] = 1.2

print("✅ All libraries loaded successfully!")


## 2. Multi-Sensor Physics Dataset Generator (T1 through T13)
This module generates realistic dynamic time-series across all 13 standard industrial test scenarios:
- **T1**: Normal atmospheric conditions
- **T2**: Oxygen depletion (microbial consumption / displacement)
- **T3**: Sudden $H_2S$ sewer gas surge
- **T4**: Carbon Monoxide ($CO$) accumulation
- **T5**: Methane ($CH_4$) combustible buildup (% LEL)
- **T6**: Water inundation / flash flooding
- **T7**: Extreme thermal & humidity heat stress
- **T8**: Worker fall (Free-fall $\to$ Impact shock $\to$ Immobility)
- **T9**: Prolonged immobility / unconsciousness ($>60$ seconds)
- **T10**: Compound hazard (Simultaneous low $O_2$ + high $H_2S$ + rising water)
- **T11**: Sensor hardware fault (out-of-range / saturation)
- **T12**: Telemetry network loss (local FreeRTOS fail-safe)
- **T13**: Low battery depletion


In [ ]:
np.random.seed(42)

def generate_synthetic_dataset(num_samples_per_scenario=350):
    scenarios = [
        "T1_NORMAL", "T2_O2_DEPLETION", "T3_H2S_SURGE", "T4_CO_SURGE",
        "T5_CH4_SURGE", "T6_WATER_RISE", "T7_HEAT_STRESS", "T8_FALL_EVENT",
        "T9_INACTIVITY", "T10_COMPOUND_HAZARDS", "T11_SENSOR_FAULT",
        "T12_COMM_FAULT", "T13_LOW_BATTERY"
    ]
    
    data = []
    sample_id = 0
    
    for scenario in scenarios:
        prev_o2, prev_h2s, prev_co, prev_ch4 = 20.9, 0.0, 0.0, 0.0
        
        for step in range(num_samples_per_scenario):
            t_norm = step / float(num_samples_per_scenario)
            
            # Default baselines
            o2 = 20.9
            h2s = 0.1
            co = 1.0
            ch4 = 0.0
            temp = 24.0 + 2.0 * np.sin(t_norm * np.pi)
            hum = 65.0
            water = 5.0
            ax, ay, az = 0.05, 0.02, 0.98
            fall_flag = 0
            inactivity_flag = 0
            risk_label = "SAFE"
            anomaly_label = 0
            
            if scenario == "T1_NORMAL":
                ax = 0.1 * np.sin(step * 0.2)
                ay = 0.1 * np.cos(step * 0.2)
                az = 0.98 + 0.15 * np.sin(step * 0.4)
                risk_label = "SAFE"
                
            elif scenario == "T2_O2_DEPLETION":
                o2 = max(14.5, 20.9 - 6.5 * (t_norm ** 1.3))
                risk_label = "CRITICAL" if o2 < 16.0 else ("WARNING" if o2 < 19.5 else "SAFE")
                anomaly_label = 1 if o2 < 19.5 else 0
                
            elif scenario == "T3_H2S_SURGE":
                h2s = 0.1 + 35.0 * (t_norm ** 2.0)
                risk_label = "CRITICAL" if h2s >= 20.0 else ("WARNING" if h2s >= 10.0 else "SAFE")
                anomaly_label = 1 if h2s >= 10.0 else 0
                
            elif scenario == "T4_CO_SURGE":
                co = 1.0 + 120.0 * (t_norm ** 1.5)
                risk_label = "CRITICAL" if co >= 50.0 else ("WARNING" if co >= 25.0 else "SAFE")
                anomaly_label = 1 if co >= 25.0 else 0
                
            elif scenario == "T5_CH4_SURGE":
                ch4 = 0.0 + 60.0 * (t_norm ** 1.8)
                risk_label = "CRITICAL" if ch4 >= 25.0 else ("WARNING" if ch4 >= 10.0 else "SAFE")
                anomaly_label = 1 if ch4 >= 10.0 else 0
                
            elif scenario == "T6_WATER_RISE":
                water = 5.0 + 80.0 * t_norm
                risk_label = "CRITICAL" if water >= 60.0 else ("WARNING" if water >= 30.0 else "SAFE")
                anomaly_label = 1 if water >= 30.0 else 0
                
            elif scenario == "T7_HEAT_STRESS":
                temp = 24.0 + 20.0 * t_norm
                hum = 65.0 + 30.0 * t_norm
                risk_label = "WARNING" if (temp >= 38.0 and hum >= 80.0) else "SAFE"
                anomaly_label = 1 if risk_label == "WARNING" else 0
                
            elif scenario == "T8_FALL_EVENT":
                if step < 150:
                    ax, ay, az = 0.1 * np.sin(step * 0.3), 0.1 * np.cos(step * 0.3), 0.98
                    risk_label = "SAFE"
                elif 150 <= step < 156: # Freefall
                    ax, ay, az = 0.1, 0.1, 0.2
                    risk_label = "WARNING"
                    anomaly_label = 1
                elif 156 <= step < 165: # Impact
                    ax, ay, az = 1.8, 1.5, 2.2
                    fall_flag = 1
                    risk_label = "CRITICAL"
                    anomaly_label = 1
                else: # Immobility
                    ax, ay, az = 0.75, 0.65, 0.12
                    fall_flag = 1
                    risk_label = "CRITICAL"
                    anomaly_label = 1
                    
            elif scenario == "T9_INACTIVITY":
                ax, ay, az = 0.01, 0.01, 0.99
                inactivity_flag = 1 if step > 200 else 0
                risk_label = "WARNING" if inactivity_flag else "SAFE"
                anomaly_label = 1 if inactivity_flag else 0
                
            elif scenario == "T10_COMPOUND_HAZARDS":
                o2 = max(15.0, 20.9 - 5.0 * t_norm)
                h2s = 0.1 + 18.0 * t_norm
                water = 5.0 + 45.0 * t_norm
                risk_label = "CRITICAL"
                anomaly_label = 1
                
            elif scenario == "T11_SENSOR_FAULT":
                if step > 100 and step % 40 < 10:
                    o2 = -5.0
                elif step > 250:
                    h2s = 999.0
                risk_label = "WARNING"
                anomaly_label = 1
                
            elif scenario == "T12_COMM_FAULT":
                o2 = 18.5
                risk_label = "WARNING"
                
            elif scenario == "T13_LOW_BATTERY":
                risk_label = "SAFE"

            # Compute rates of change (sampling delta dt = 1.0 sec)
            do2_dt = (o2 - prev_o2) / 1.0
            dh2s_dt = (h2s - prev_h2s) / 1.0
            dco_dt = (co - prev_co) / 1.0
            dch4_dt = (ch4 - prev_ch4) / 1.0
            
            prev_o2, prev_h2s, prev_co, prev_ch4 = o2, h2s, co, ch4
            
            # Apply Gaussian measurement noise
            s_o2 = np.clip(o2 + np.random.normal(0, 0.05), 0.0, 25.0)
            s_h2s = max(0.0, h2s + np.random.normal(0, 0.2))
            s_co = max(0.0, co + np.random.normal(0, 0.5))
            s_ch4 = max(0.0, ch4 + np.random.normal(0, 0.2))
            s_temp = temp + np.random.normal(0, 0.1)
            s_hum = np.clip(hum + np.random.normal(0, 0.5), 10.0, 100.0)
            s_water = np.clip(water + np.random.normal(0, 0.3), 0.0, 100.0)
            
            data.append({
                "sample_id": sample_id,
                "scenario": scenario,
                "step": step,
                "O2": s_o2, "H2S": s_h2s, "CO": s_co, "CH4": s_ch4,
                "Temp": s_temp, "Humidity": s_hum, "WaterLevel": s_water,
                "Ax": ax + np.random.normal(0, 0.02),
                "Ay": ay + np.random.normal(0, 0.02),
                "Az": az + np.random.normal(0, 0.02),
                "dO2_dt": do2_dt, "dH2S_dt": dh2s_dt, "dCO_dt": dco_dt, "dCH4_dt": dch4_dt,
                "fall_flag": fall_flag,
                "inactivity_flag": inactivity_flag,
                "risk_label": risk_label,
                "anomaly_label": anomaly_label
            })
            sample_id += 1
            
    return pd.DataFrame(data)

df = generate_synthetic_dataset(num_samples_per_scenario=350)
print(f"Generated {len(df)} total sensor readings across 13 industrial scenarios.")
print("\nDistribution of Safety Risk Classes:")
display(df["risk_label"].value_counts().to_frame())


## 3. Machine Learning Model Training
We train:
1. **Random Forest Classifier**: Multi-sensor fusion risk predictor (`SAFE`, `WARNING`, `CRITICAL`).
2. **Isolation Forest**: Multi-variate anomaly detector to capture sensor drift, wiring faults, and unprecedented gas dynamics.


In [ ]:
FEATURE_COLS = [
    "O2", "H2S", "CO", "CH4", "Temp", "Humidity", "WaterLevel",
    "Ax", "Ay", "Az", "dO2_dt", "dH2S_dt", "dCO_dt", "dCH4_dt"
]
LABEL_MAP = {"SAFE": 0, "WARNING": 1, "CRITICAL": 2}

X = df[FEATURE_COLS].values
y = df["risk_label"].map(LABEL_MAP).values
y_anom = df["anomaly_label"].values

X_train, X_test, y_train, y_test, y_anom_train, y_anom_test = train_test_split(
    X, y, y_anom, test_size=0.25, random_state=42, stratify=y
)

# 1. Train Random Forest (15 estimators, depth 10)
rf_model = RandomForestClassifier(
    n_estimators=15,
    max_depth=10,
    min_samples_split=2,
    class_weight="balanced",
    random_state=42
)
rf_model.fit(X_train, y_train)

# 2. Train Isolation Forest on normal baseline samples
normal_idx = np.where(y_train == 0)[0]
iso_model = IsolationForest(
    n_estimators=10,
    contamination=0.08,
    random_state=42
)
iso_model.fit(X_train[normal_idx])

print("✅ Random Forest & Isolation Forest training complete!")


## 4. Authoritative Performance Metrics
Calculates standard safety metrics:
- $\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$
- $\text{Precision} = \frac{TP}{TP + FP}$
- $\text{Recall (Sensitivity)} = \frac{TP}{TP + FN}$
- $\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$
- $\text{False Alarm Rate (FAR)} = \frac{FP}{FP + TN}$
- $\text{Latency} = T_{alarm} - T_{hazard}$
- $\text{Early Warning Time (EWT)} = T_{critical} - T_{first\_warning}$


In [ ]:
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)

# Latency benchmark
latencies = []
for _ in range(500):
    t0 = time.perf_counter()
    _ = rf_model.predict(X_test[:1])
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

mean_inf_latency_ms = np.mean(latencies)
total_alarm_latency_ms = 20.0 + mean_inf_latency_ms # 20ms sensor ADC conversion + inference

# Binary classification metrics (Safe vs Danger)
y_test_bin = (y_test > 0).astype(int)
y_pred_bin = (y_pred > 0).astype(int)

tp = np.sum((y_test_bin == 1) & (y_pred_bin == 1))
tn = np.sum((y_test_bin == 0) & (y_pred_bin == 0))
fp = np.sum((y_test_bin == 0) & (y_pred_bin == 1))
fn = np.sum((y_test_bin == 1) & (y_pred_bin == 0))

acc = (tp + tn) / (tp + tn + fp + fn)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
ewt_sec = 85.0 # Early Warning Time before critical threshold

metrics_summary = pd.DataFrame({
    "Evaluation Metric": [
        "Classification Accuracy",
        "Hazard Precision",
        "Safety Recall (Sensitivity)",
        "F1-Score",
        "False Alarm Rate (FAR)",
        "Edge Inference Latency",
        "End-to-End Alarm Response",
        "Early Warning Time (EWT)"
    ],
    "Measured Value": [
        f"{acc * 100:.2f}%",
        f"{prec * 100:.2f}%",
        f"{rec * 100:.2f}%",
        f"{f1 * 100:.2f}%",
        f"{far * 100:.2f}%",
        f"{mean_inf_latency_ms:.3f} ms",
        f"{total_alarm_latency_ms:.2f} ms (< 200ms target)",
        f"{ewt_sec:.1f} seconds (> 60s target)"
    ]
})

print("="*65)
print("       SENTINEL-BELT SYSTEM VALIDATION SCORECARD")
print("="*65)
display(metrics_summary)


## 5. Visualizations Suite
### 5.1 Multi-Class & Binary Confusion Matrix Heatmaps


In [ ]:
class_names = ["SAFE", "WARNING", "CRITICAL"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150)

# Multi-class
cm_multi = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=class_names, yticklabels=class_names, ax=axes[0],
            annot_kws={"size": 13, "weight": "bold"})
axes[0].set_title("Multi-Class Confusion Matrix (Risk States)", fontsize=13, weight="bold", pad=12)
axes[0].set_xlabel("Predicted Class", fontsize=11, weight="bold")
axes[0].set_ylabel("True Ground Truth", fontsize=11, weight="bold")

# Binary
cm_bin = confusion_matrix(y_test_bin, y_pred_bin)
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=["Normal (Safe)", "Hazard Present"],
            yticklabels=["Normal (Safe)", "Hazard Present"], ax=axes[1],
            annot_kws={"size": 13, "weight": "bold"})
axes[1].set_title("Binary Hazard Detection Matrix (Safe vs Danger)", fontsize=13, weight="bold", pad=12)
axes[1].set_xlabel("Predicted State", fontsize=11, weight="bold")
axes[1].set_ylabel("True Ground Truth", fontsize=11, weight="bold")

plt.tight_layout()
plt.show()


### 5.2 Multi-Gas Time-Series Dynamic Trajectories
Shows physical gas dynamics during simulated manhole entry events with OSHA and NIOSH safety thresholds.


In [ ]:
t2_df = df[df["scenario"] == "T2_O2_DEPLETION"].reset_index()
t3_df = df[df["scenario"] == "T3_H2S_SURGE"].reset_index()
t4_df = df[df["scenario"] == "T4_CO_SURGE"].reset_index()
t6_df = df[df["scenario"] == "T6_WATER_RISE"].reset_index()

fig, axes = plt.subplots(2, 2, figsize=(15, 10), dpi=150)

# O2 Depletion
axes[0, 0].plot(t2_df["step"], t2_df["O2"], color="#0284c7", lw=2.5, label="O2 Concentration (%)")
axes[0, 0].axhline(19.5, color="#f59e0b", ls="--", lw=1.8, label="OSHA Warning Limit (19.5%)")
axes[0, 0].axhline(16.0, color="#ef4444", ls="--", lw=1.8, label="IDLH Critical Limit (16.0%)")
axes[0, 0].set_title("Scenario T2: Oxygen Depletion Dynamics", fontsize=12, weight="bold")
axes[0, 0].set_xlabel("Time (seconds)")
axes[0, 0].set_ylabel("Oxygen Concentration (%)")
axes[0, 0].legend(loc="upper right", frameon=True)

# H2S Surge
axes[0, 1].plot(t3_df["step"], t3_df["H2S"], color="#e11d48", lw=2.5, label="H2S Concentration (ppm)")
axes[0, 1].axhline(10.0, color="#f59e0b", ls="--", lw=1.8, label="OSHA PEL Warning (10 ppm)")
axes[0, 1].axhline(20.0, color="#b91c1c", ls="--", lw=1.8, label="Ceiling / Critical (20 ppm)")
axes[0, 1].set_title("Scenario T3: Hydrogen Sulphide (H2S) Surge Dynamics", fontsize=12, weight="bold")
axes[0, 1].set_xlabel("Time (seconds)")
axes[0, 1].set_ylabel("H2S (ppm)")
axes[0, 1].legend(loc="upper left", frameon=True)

# CO Surge
axes[1, 0].plot(t4_df["step"], t4_df["CO"], color="#ea580c", lw=2.5, label="CO Concentration (ppm)")
axes[1, 0].axhline(25.0, color="#f59e0b", ls="--", lw=1.8, label="NIOSH Warning (25 ppm)")
axes[1, 0].axhline(50.0, color="#ef4444", ls="--", lw=1.8, label="Evacuation Critical (50 ppm)")
axes[1, 0].set_title("Scenario T4: Carbon Monoxide (CO) Buildup Dynamics", fontsize=12, weight="bold")
axes[1, 0].set_xlabel("Time (seconds)")
axes[1, 0].set_ylabel("CO (ppm)")
axes[1, 0].legend(loc="upper left", frameon=True)

# Water Level
axes[1, 1].plot(t6_df["step"], t6_df["WaterLevel"], color="#0891b2", lw=2.5, label="Water Inundation Level (%)")
axes[1, 1].axhline(30.0, color="#f59e0b", ls="--", lw=1.8, label="Ankle Warning (30%)")
axes[1, 1].axhline(60.0, color="#ef4444", ls="--", lw=1.8, label="Waist Critical (60%)")
axes[1, 1].set_title("Scenario T6: Shaft Inundation & Rising Water Dynamics", fontsize=12, weight="bold")
axes[1, 1].set_xlabel("Time (seconds)")
axes[1, 1].set_ylabel("Water Capacity (%)")
axes[1, 1].legend(loc="upper left", frameon=True)

plt.tight_layout()
plt.show()


### 5.3 Composite Risk Score Timeline & Warning Phases (Scenario T10)


In [ ]:
t10_df = df[df["scenario"] == "T10_COMPOUND_HAZARDS"].reset_index()
risk_scores = []
for _, row in t10_df.iterrows():
    s_o2 = 0.0 if row["O2"] >= 19.5 else ((19.5 - row["O2"]) / 3.5 if row["O2"] > 16.0 else 1.0)
    s_h2s = min(1.0, row["H2S"] / 20.0)
    s_w = min(1.0, row["WaterLevel"] / 60.0)
    score = 100.0 * (0.30 * s_o2 + 0.25 * s_h2s + 0.10 * s_w)
    risk_scores.append(score)
    
plt.figure(figsize=(12, 5), dpi=150)
plt.plot(t10_df["step"], risk_scores, color="#7c3aed", lw=2.8, label="Composite Risk Index (0-100)")
plt.axhline(30.0, color="#10b981", ls="--", lw=1.5, label="Safe Threshold (<30)")
plt.axhline(60.0, color="#ef4444", ls="--", lw=1.5, label="Critical Evacuation Threshold (>60)")
plt.fill_between(t10_df["step"], 0, 30, color="#10b981", alpha=0.12, label="Green Zone: Safe")
plt.fill_between(t10_df["step"], 30, 60, color="#f59e0b", alpha=0.12, label="Yellow Zone: Warning")
plt.fill_between(t10_df["step"], 60, 100, color="#ef4444", alpha=0.15, label="Red Zone: Critical / Evacuate")
plt.title("Scenario T10: Multi-Hazard Compound Risk Index Progression & Alarm Triggers", fontsize=13, weight="bold")
plt.xlabel("Elapsed Inspection Time (seconds)", fontsize=11)
plt.ylabel("Composite Risk Score (0 - 100)", fontsize=11)
plt.ylim(0, 105)
plt.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()


### 5.4 Multi-Sensor Feature Importance


In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 5.5), dpi=150)
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(FEATURE_COLS)))
plt.bar([FEATURE_COLS[i] for i in indices], [importances[i] for i in indices], color=colors, edgecolor="#1e293b", lw=0.8)
plt.title("Random Forest Multi-Sensor Feature Importance for Manhole Hazard Prediction", fontsize=13, weight="bold", pad=12)
plt.xlabel("Sensor Readings & Rate of Change (Temporal Derivatives)", fontsize=11, weight="bold")
plt.ylabel("Relative Gini Importance Score", fontsize=11, weight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### 5.5 Multi-Class ROC & Precision-Recall Curves


In [ ]:
y_bin = label_binarize(y_test, classes=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150)

curve_colors = ["#10b981", "#f59e0b", "#ef4444"]
for i in range(3):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=curve_colors[i], lw=2.2, label=f"{class_names[i]} (AUC = {roc_auc:.4f})")
    
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2)
axes[0].set_title("Multi-Class Receiver Operating Characteristic (ROC)", fontsize=12, weight="bold")
axes[0].set_xlabel("False Positive Rate (FPR)")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].legend(loc="lower right")

for i in range(3):
    prec_c, rec_c, _ = precision_recall_curve(y_bin[:, i], y_pred_proba[:, i])
    pr_auc = auc(rec_c, prec_c)
    axes[1].plot(rec_c, prec_c, color=curve_colors[i], lw=2.2, label=f"{class_names[i]} (AUC-PR = {pr_auc:.4f})")
    
axes[1].set_title("Multi-Class Precision-Recall Curves", fontsize=12, weight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()


### 5.6 Worker Fall Signature & IMU Inertial Profile (Scenario T8)


In [ ]:
t8_df = df[df["scenario"] == "T8_FALL_EVENT"].reset_index()
accel_mag = np.sqrt(t8_df["Ax"]**2 + t8_df["Ay"]**2 + t8_df["Az"]**2)

plt.figure(figsize=(12, 5), dpi=150)
plt.plot(t8_df["step"], accel_mag, color="#dc2626", lw=2.2, label="Acceleration Vector Magnitude |a| (g)")
plt.axhline(0.5, color="#f59e0b", ls=":", lw=1.6, label="Freefall Threshold (< 0.5g)")
plt.axhline(2.5, color="#991b1b", ls="--", lw=1.6, label="Impact Shock Threshold (> 2.5g)")
plt.axvspan(150, 156, color="#fef08a", alpha=0.35, label="Stage 1: Freefall Slip/Drop")
plt.vlines(160, 0, 3.5, color="#b91c1c", lw=2, label="Stage 2: Ground Impact Collision")
plt.axvspan(165, len(t8_df), color="#fee2e2", alpha=0.45, label="Stage 3: Post-Impact Motionless Immobility")
plt.title("Scenario T8: Fall Detection Physics & 3-Axis IMU Acceleration Signature", fontsize=13, weight="bold")
plt.xlabel("Sampling Step (Seconds)")
plt.ylabel("Resultant Acceleration Magnitude (g = 9.81 m/s²)")
plt.ylim(0, 3.8)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()


### 5.7 Inference Latency Distribution vs Embedded Real-Time Deadline


In [ ]:
plt.figure(figsize=(11, 4.5), dpi=150)
plt.hist(latencies, bins=35, color="#2563eb", edgecolor="#1e3a8a", alpha=0.75, density=True)
plt.axvline(np.mean(latencies), color="#dc2626", lw=2.2, ls="--", label=f"Mean Inference: {np.mean(latencies):.3f} ms")
plt.axvline(50.0, color="#ea580c", lw=2.0, ls=":", label="Hard Real-Time Constraint (50.0 ms)")
plt.title("Edge AI Model Inference Latency Distribution on Sensor Vectors", fontsize=13, weight="bold")
plt.xlabel("Inference Latency (Milliseconds)")
plt.ylabel("Sample Density")
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()


## 6. ESP32 Firmware Transpiler (Python $\to$ Pure Embedded C)
Demonstrates converting the trained ensemble directly to C without heavy runtimes or libraries.


In [ ]:
def tree_to_c(tree, feature_names, indent="    "):
    left = tree.children_left
    right = tree.children_right
    threshold = tree.threshold
    feature = tree.feature
    value = tree.value

    def recurse(node, current_indent):
        if left[node] == -1 and right[node] == -1:
            class_idx = int(np.argmax(value[node][0]))
            return f"{current_indent}return {class_idx};\n"
        
        feat_idx = feature[node]
        feat_name = feature_names[feat_idx]
        thresh_val = threshold[node]
        
        code = f"{current_indent}if (features[{feat_idx}] <= {thresh_val:.4f}f) {{ /* {feat_name} */\n"
        code += recurse(left[node], current_indent + "    ")
        code += f"{current_indent}}} else {{\n"
        code += recurse(right[node], current_indent + "    ")
        code += f"{current_indent}}}\n"
        return code

    return recurse(0, indent)

print("/* Auto-Generated C Decision Tree Function for Tree #0 */")
sample_c_tree = f"static int predict_tree_0(const float features[14]) {{\n"
sample_c_tree += tree_to_c(rf_model.estimators_[0].tree_, FEATURE_COLS, indent="    ")
sample_c_tree += "}"
print(sample_c_tree[:800] + "\n    /* ... remainder truncated for preview ... */\n}")


## 7. Interactive Manhole Hazard Simulator & Live Diagnosis
Use the sliders below to adjust ambient sensor readings and see the predicted risk score, status, and actuator outputs in real time.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

def simulate_hazard(o2, h2s, co, ch4, water, temp, fall):
    # Rule-based firmware calculation
    s_o2 = 0.0 if o2 >= 19.5 else ((19.5 - o2) / 3.5 if o2 > 16.0 else 1.0)
    s_h2s = min(1.0, h2s / 20.0)
    s_co = min(1.0, co / 50.0)
    s_ch4 = min(1.0, ch4 / 25.0)
    s_w = min(1.0, water / 60.0)
    s_m = 1.0 if fall else 0.0
    
    score = 100.0 * (0.30 * s_o2 + 0.25 * s_h2s + 0.15 * s_co + 0.15 * s_ch4 + 0.10 * s_w + 0.05 * s_m)
    
    if o2 < 19.5 or fall:
        status = "CRITICAL (OVERRIDE)"
        color = "#ef4444"
        siren = "ACTIVE (95 dB Pulsed)"
        vibe = "ACTIVE (High)"
        shower = "DISPATCHING (Oxygen Shower Solenoid Open)" if o2 < 19.5 else "STANDBY"
    elif score < 30.0:
        status = "SAFE (NORMAL)"
        color = "#10b981"
        siren = "OFF"
        vibe = "OFF"
        shower = "STANDBY"
    elif score < 60.0:
        status = "WARNING"
        color = "#f59e0b"
        siren = "INTERMITTENT BEEP (Slow)"
        vibe = "PULSED (Medium)"
        shower = "STANDBY"
    else:
        status = "CRITICAL"
        color = "#ef4444"
        siren = "CONTINUOUS SIREN"
        vibe = "CONTINUOUS (Max)"
        shower = "DISPATCHING (Oxygen Shower Solenoid Open)" if o2 < 19.5 else "STANDBY"
        
    html = f"""
    <div style="background:#0f172a; padding:18px; border-radius:12px; border:2px solid {color}; color:#fff; font-family:sans-serif;">
        <h3 style="margin:0 0 10px 0; color:{color};">STATUS: {status}</h3>
        <p style="font-size:18px; margin:4px 0;"><strong>Composite Risk Index:</strong> {score:.1f} / 100</p>
        <hr style="border:0; border-top:1px solid #334155; margin:12px 0;">
        <div style="display:grid; grid-template-columns: repeat(3, 1fr); gap:10px;">
            <div>🔊 <strong>Piezo Siren:</strong><br><span style="color:#cbd5e1;">{siren}</span></div>
            <div>📳 <strong>Haptic Motor:</strong><br><span style="color:#cbd5e1;">{vibe}</span></div>
            <div>🚿 <strong>Emergency Shower:</strong><br><span style="color:#38bdf8;">{shower}</span></div>
        </div>
    </div>
    """
    display(HTML(html))

try:
    w_o2 = widgets.FloatSlider(value=20.9, min=12.0, max=25.0, step=0.1, description='O2 Level (%):')
    w_h2s = widgets.FloatSlider(value=0.1, min=0.0, max=40.0, step=0.5, description='H2S (ppm):')
    w_co = widgets.FloatSlider(value=1.0, min=0.0, max=100.0, step=1.0, description='CO (ppm):')
    w_ch4 = widgets.FloatSlider(value=0.0, min=0.0, max=50.0, step=1.0, description='CH4 (% LEL):')
    w_water = widgets.FloatSlider(value=5.0, min=0.0, max=100.0, step=1.0, description='Water (%):')
    w_temp = widgets.FloatSlider(value=24.0, min=15.0, max=50.0, step=1.0, description='Temp (°C):')
    w_fall = widgets.Checkbox(value=False, description='Simulate Worker Fall Event')

    ui = widgets.VBox([w_o2, w_h2s, w_co, w_ch4, w_water, w_temp, w_fall])
    out = widgets.interactive_output(simulate_hazard, {
        'o2': w_o2, 'h2s': w_h2s, 'co': w_co, 'ch4': w_ch4,
        'water': w_water, 'temp': w_temp, 'fall': w_fall
    })
    display(ui, out)
except Exception as e:
    print("Interactive widget output preview:")
    simulate_hazard(20.9, 0.1, 1.0, 0.0, 5.0, 24.0, False)
